# All tasks on one interactive map: UMAP and PCA

Where do the seven ambulation tasks sit relative to one another **in the raw sensor space**, before
any model learns a representation? This notebook embeds standardized sensor windows with UMAP and
lets you pan, zoom and rotate through them in 2-D and 3-D. It is a browsing tool, not a benchmark:
nothing is trained, scored or thresholded here, and no task is labelled in-distribution or
out-of-distribution.

Learned representations have their own UMAP sections: the AE latent map in
[02_autoencoder_lof.ipynb](02_autoencoder_lof.ipynb) and the VAE posterior means, samples and
variances in [08_vae_variants.ipynb](08_vae_variants.ipynb).

In [1]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'ml/sci_rob/vanilla/registry.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter within this repository.')
sys.path.insert(0, str(ROOT / 'notebooks/sci_rob/model_exploration'))
get_ipython().run_line_magic('matplotlib', 'inline')
from _exploration import *
import torch
from dataclasses import asdict
torch.set_num_threads(2)  # small architecture checks; training CLI runs separately
torch.manual_seed(0)
RUN_TRAINING = False
print('Repository:', ROOT)
print('Kernel:', sys.executable)

Repository: /Users/dariussattari/Documents/thesis/Uncertainty-Aware-Hip-Exoskeleton
Kernel: /Users/dariussattari/Documents/thesis/Uncertainty-Aware-Hip-Exoskeleton/.venv/bin/python


## What is embedded

- Source: the mode-complete set `data/processed_all/AE_GAN` (all seven tasks in one split, the same
  windows the LOTO runs score). `SPLIT` defaults to the test participants; `train` and `val` also work
  but take longer to read.
- Each window is 16 channels × 200 samples, standardized with that set's scaler (fitted on LG/RA/RD
  training windows only, so scaling does not depend on which task is held out), then flattened to
  3,200 values.
- A stratified sample of at most `MAX_POINTS` windows is drawn; standing (621 test windows) and
  transitions keep every window. The sample is reduced to 50 principal components (the standard UMAP
  preprocessing) and UMAP is fitted on all sampled windows, since there is no reference/other split here.
  The PCA panel shows the first of those components, labelled with their share of the raw window variance.

$$x\in\mathbb R^{16\times 200}\;\xrightarrow{\text{standardize, flatten}}\;\mathbb R^{3200}
\;\xrightarrow{\text{PCA}}\;\mathbb R^{50}\;\xrightarrow{\text{UMAP}}\;\mathbb R^{2}\text{ or }\mathbb R^{3}.$$

**Three projections of the same rows.** `umap` is nonlinear and neighbourhood-preserving:
distances, cluster areas and gaps are not measurements. `pca` is linear, fitted on the same
reference rows, and its axis labels carry the explained variance. `latent` is two or three raw
latent coordinates with no transformation at all: the only view in which a distance is a true
latent distance, at the price of showing 2-3 of 32 dimensions (`LATENT_DIMS = None` picks the
highest-variance ones on the reference rows; give a tuple such as `(0, 8, 16)` to choose).

**Layout.** In 2-D and in 3-D the projections are side-by-side panels of one figure (the VAE's
view buttons switch all panels together). Every 3-D panel is a WebGL scene, browsers allow only
about 8-16 live WebGL contexts per page, and JupyterLab is one page for every open notebook: a
three-panel 3-D figure costs three. If a 3-D panel ever shows its legend but a blank scene, close
the other notebook tabs and reload the page. 2-D maps are SVG and never hit that cap.

**How to move around (Plotly):** scroll to zoom, drag to pan (2-D) or rotate (3-D; right-drag
pans), double-click to reset. Click a task in the legend to hide it in every panel; double-click
a task to isolate it. Hover shows the participant and trial. `VIEW_DIMS` selects `(2,)`, `(3,)`
or `(2, 3)`; `PROJECTIONS` selects which projections appear. With `SAVE_HTML = True` each figure
is also written as a standalone page under `notebooks/sci_rob/model_exploration/interactive/` (ignored
by git) for full-screen browsing.

Every task keeps **one fixed colour and marker symbol**, whatever role a protocol assigns it.
UMAP embeddings are cached under `.cache/`; delete that folder to recompute.

In [2]:
from _umap_exploration import sensor_windows, feature_views, task_counts, TASK_COLORS
SPLIT = 'test'               # 'train', 'val' or 'test' participants
VIEW_DIMS = (2, 3)           # (2,), (3,) or (2, 3)
PROJECTIONS = ('umap', 'pca')  # no latent space here: raw windows have 3,200 coordinates
MAX_POINTS = 12_000          # stratified sample of windows; small tasks keep every window
VIEW_SEED = 0
SAVE_HTML = True             # standalone copies under notebooks/sci_rob/model_exploration/interactive/
windows = sensor_windows(SPLIT, max_points=MAX_POINTS, seed=VIEW_SEED)
print(f"{len(windows['meta']):,} of the {SPLIT} windows sampled; 50 PCs keep "
      f"{windows['explained']:.1%} of the variance of the flattened windows.")
display(task_counts(windows['meta']))
all_task_views = feature_views(windows['features'], windows['meta'], np.arange(len(windows['meta'])),
                               dims=VIEW_DIMS, projections=PROJECTIONS, title=f'Sensor windows ({SPLIT})',
                               cache_key=f'sensor_{SPLIT}', seed=VIEW_SEED, max_points=None,
                               save_html=SAVE_HTML, fitted_on='all sampled windows',
                               pca_variance_scale=windows['explained'])

15,361 of the test windows sampled; 50 PCs keep 74.0% of the variance of the flattened windows.


,name,windows
task,,
LG,level ground,4047
RA,ramp ascent,2612
RD,ramp descent,2087
SA,stair ascent,2000
SD,stair descent,2000
TR,transitions,1994
ST,standing,621


Standalone interactive copy: notebooks/sci_rob/model_exploration/interactive/sensor_test_views_2d.html


Standalone interactive copy: notebooks/sci_rob/model_exploration/interactive/sensor_test_views_3d.html


## Reading the map

- Overlap between tasks in this map means their one-second sensor windows look alike to a
  neighbourhood-preserving embedding of PCA features. It does not mean a detector cannot separate
  them, and separation here does not mean a detector will.
- Windows from one trial are ten samples apart and overlap heavily, so strands of points often trace
  single trials. Hover to see which participant and trial a strand belongs to; isolate a task with a
  double-click on the legend.
- Compare with the AE latent map and the VAE posterior views to see what each model keeps or discards.
- UMAP is stochastic and seed-dependent; `VIEW_SEED` and the cache make a view reproducible, not
  canonical. Distances and cluster areas are not measurements; the PCA panel is the linear reference.

## Variant: does the pelvis IMU add task structure?

The raw trials carry a seventh sensor the processed set leaves out: a pelvis-mounted IMU
(`pelvis_accel_xyz`, `pelvis_gyro_xyz`, six channels in `exo.csv`). Adding it to the model input
is not free: few public gait datasets carry a pelvis IMU at the same placement, so every channel
added here narrows the datasets a trained model can later be checked against. Before paying that,
look at the raw data alone: does the pelvis IMU change how the seven tasks sit relative to one another?

Three channel sets are built from **the same sampled windows** (same rows, same order):

| set | channels | standardization |
|---|---|---|
| `thigh` | the processed 16: hip encoder angle + velocity and thigh IMU, both legs | the set's scaler (LG/RA/RD training windows) |
| `pelvis` | the 6 pelvis IMU channels, sliced from the raw trials at the same `end_idx` | mean/std over the same LG/RA/RD training trials |
| `thigh+pelvis` | all 22 | both |

Each set is flattened, reduced to 50 principal components and embedded exactly as above. The
`thigh` views are the figures higher up; the two new sets get their own maps below.

Looking is not enough to say "worth adding", so one number accompanies the maps: **k-NN task accuracy**
(k = 15) in the 50-PC space, and in the UMAP embeddings for reference. Windows from one trial overlap by
95 % and would label their own neighbours under a random split, so whole recordings are held out, in two ways:

- *participant held out*: one fold per test participant, the stricter question (does the structure carry
  across people?). Standing and transitions were recorded by one test participant only, so they cannot be
  scored this way (recall NaN) and they pull the plain accuracy down for every set alike.
- *trial held out*: five folds of whole trials. Transitions (five trials) are scored here; standing is a
  single trial in the test split, so it stays unscored under both schemes and its 621 windows count only
  as a distractor the other tasks must not be confused with.

The PCA-space columns are the ones to read; UMAP is stochastic and its silhouettes reward the tight strands
it draws. A set whose accuracy and per-task recall do not move has added channels without adding task
structure in the raw space; that does not rule out a learned model finding something, but it is the cheap check.

In [3]:
from _umap_exploration import sensor_window_sets, separability, FEATURE_SETS
sets = sensor_window_sets(SPLIT, tuple(FEATURE_SETS), max_points=MAX_POINTS, seed=VIEW_SEED)
assert np.array_equal(sets['thigh']['source_index'], windows['source_index']), 'the variant must reuse the rows plotted above'
for name, s in sets.items():
    print(f"{name:13s} {len(s['channels']):2d} channels; 50 PCs keep {s['explained']:.1%} of the flattened-window variance")
if sets['thigh']['dropped']:
    print(f"{sets['thigh']['dropped']} windows dropped from every set: NaN in their pelvis channels")
variant_views = {}
for name in ('pelvis', 'thigh+pelvis'):
    s = sets[name]
    variant_views[name] = feature_views(s['features'], s['meta'], np.arange(len(s['meta'])),
                                        dims=VIEW_DIMS, projections=PROJECTIONS,
                                        title=f'Sensor windows ({SPLIT}), {name}',
                                        cache_key=f"sensor_{SPLIT}_{name.replace('+', '_')}", seed=VIEW_SEED,
                                        max_points=None, save_html=SAVE_HTML, fitted_on='all sampled windows',
                                        pca_variance_scale=s['explained'])

thigh         16 channels; 50 PCs keep 74.0% of the flattened-window variance
pelvis         6 channels; 50 PCs keep 82.8% of the flattened-window variance
thigh+pelvis  22 channels; 50 PCs keep 72.5% of the flattened-window variance


Standalone interactive copy: notebooks/sci_rob/model_exploration/interactive/sensor_test_pelvis_views_2d.html


Standalone interactive copy: notebooks/sci_rob/model_exploration/interactive/sensor_test_pelvis_views_3d.html


Standalone interactive copy: notebooks/sci_rob/model_exploration/interactive/sensor_test_thigh_pelvis_views_2d.html


Standalone interactive copy: notebooks/sci_rob/model_exploration/interactive/sensor_test_thigh_pelvis_views_3d.html


### Task separability by channel set

Rows are (channel set, held-out unit); the first two columns are the numbers to read. `per_task` gives the
PCA-space recall of each task under each scheme, so a gain (or loss) can be traced to the tasks it comes from.

In [4]:
views = {'thigh': all_task_views, **variant_views}
summary, per_task = separability({name: dict(features=sets[name]['features'],
                                             umap={d: v[d]['points']['umap'] for d in v})
                                  for name, v in views.items()}, sets['thigh']['meta'], k=15, seed=VIEW_SEED)
display(summary.round(3))
display(per_task.round(3))

kNN accuracy, PCA-50  balanced accuracy, PCA  \
feature set  held out                                                    
thigh        participant                 0.600                   0.708   
             trial                       0.669                   0.705   
pelvis       participant                 0.496                   0.530   
             trial                       0.627                   0.657   
thigh+pelvis participant                 0.615                   0.707   
             trial                       0.685                   0.719   

                          kNN accuracy, UMAP-2D  silhouette, UMAP-2D  \
feature set  held out                                                  
thigh        participant                  0.566               -0.028   
             trial                        0.662               -0.028   
pelvis       participant                  0.399               -0.107   
             trial                        0.592               -0.107   
thigh+pelvis participant                  0.586               -0.003   
             trial                        0.696               -0.003   

                          kNN accuracy, UMAP-3D  silhouette, UMAP-3D  
feature set  held out                                                 
thigh        participant                  0.629               -0.043  
             trial                        0.689               -0.043  
pelvis       participant                  0.444               -0.056  
             trial                        0.647               -0.056  
thigh+pelvis participant                  0.618                0.002  
             trial                        0.711                0.002

held out,participant,trial,participant,trial,participant,trial
feature set,thigh,thigh,pelvis,pelvis,thigh+pelvis,thigh+pelvis
task,,,,,,
LG,0.775,0.660,0.879,0.634,0.878,0.695
RA,0.791,0.672,0.797,0.647,0.849,0.666
RD,0.723,0.735,0.338,0.696,0.690,0.755
SA,0.564,0.812,0.222,0.680,0.457,0.801
SD,0.689,0.852,0.411,0.750,0.662,0.857
TR,NaN,0.497,NaN,0.532,NaN,0.541
ST,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
source_links(['ml/sci_rob/LOTO/splits.py', 'ml/sci_rob/LOTO/build_data.py', 'ml/sci_rob/vanilla/dataset.py', 'notebooks/sci_rob/model_exploration/_umap_exploration.py'])

Implementation sources: [splits.py](../../../ml/sci_rob/LOTO/splits.py), [build_data.py](../../../ml/sci_rob/LOTO/build_data.py), [dataset.py](../../../ml/sci_rob/vanilla/dataset.py), [_umap_exploration.py](../../../notebooks/sci_rob/model_exploration/_umap_exploration.py)